In [2]:
%load_ext autoreload
%autoreload 2

# Import Libraries

In [ ]:
from pathlib import Path
import shutil

import pandas as pd
from sklearn.model_selection import train_test_split

from mra_midas_skin_cancer_ml.utils.process_metadata import (
    create_lesion_key,
    dedupe_metadata,
    drop_na_target_img,
    import_metadata,
    process_target,
    get_data_dir,
    export_metadata,
)

from mra_midas_skin_cancer_ml.utils.validate_data import (
    check_split_ratios,
    count_files_in_image_folders,
    check_master_split,
)

# Split Data Based On Image Distance

In [9]:
def process_metadata_for_img():
    """Process metadata for image splitting."""

    meta_df = import_metadata()
    meta_df = process_target(meta_df)
    meta_df = drop_na_target_img(meta_df)
    dedupe_df = dedupe_metadata(meta_df)

    meta_df = create_lesion_key(meta_df)
    merge_df = pd.merge(
        meta_df,
        dedupe_df[["lesion_key", "midas_path_binary"]],
        on=["lesion_key", "midas_path_binary"],
        how="inner",
    )

    return merge_df


def split_metadata_dist():
    """Split metadata based on image distance ("1ft", "6in", "dscope")."""

    meta_df = process_metadata_for_img()

    dist_dict = {}

    cols = [
        "lesion_key",
        "midas_record_id",
        "midas_file_name",
        "matched_file",
        "midas_path_binary",
    ]

    total_unique_lesions = set()

    for dist in ["1ft", "6in", "dscope"]:
        subset_df = meta_df[meta_df["midas_distance"] == dist]
        subset_df = subset_df[cols]

        # Drop multiple images per lesion
        subset_df = subset_df.drop_duplicates(
            subset=["lesion_key", "midas_path_binary"]
        )

        print(f"{dist} is unique: {subset_df['lesion_key'].is_unique}")
        unique_count = subset_df["lesion_key"].nunique()
        print(f"{dist} unique lesions: {unique_count} \n")

        total_unique_lesions.update(subset_df["lesion_key"].tolist())

        dist_dict[dist] = subset_df

    print(f"Total unique lesions: {len(total_unique_lesions)}\n")

    return dist_dict


dist_dict = split_metadata_dist()

Total missing files: 12
No file found: s-prd-502892079.jpg
No file found: s-prd-539536718.jpg
No file found: s-prd-539536620.jpg
No file found: s-prd-709811242.jpeg
No file found: s-prd-656881902.jpg
No file found: s-prd-656882615.jpg
No file found: s-prd-656882465.jpg
No file found: s-prd-675941199.jpeg
No file found: s-prd-692721767.jpeg
No file found: s-prd-722591153.jpeg
No file found: s-prd-722591152.jpeg
No file found: s-prd-798621909.jpg

Is unique: True
Unique count: 1035 

1ft is unique: True
1ft unique lesions: 1021 

6in is unique: True
6in unique lesions: 1028 

dscope is unique: True
dscope unique lesions: 1028 

Total unique lesions: 1035



# Split Each Image Set Into Train/Val/Test

In [ ]:
def split_train_test_by_lesion(dist_dict, test_size=0.2, random_state=42):
    """
    Split dataframes for each image distance into train/val/test sets by
    midas_record_id (patient id).
    """
    # Get all unique patient ids across all distances
    unique_ids = pd.concat(
        [df["midas_record_id"] for df in dist_dict.values()]
    ).unique()

    # Split patient_ids into train, val and test
    train_ids, val_test_ids = train_test_split(
        unique_ids, test_size=test_size, random_state=random_state
    )

    val_ids, _ = train_test_split(
        val_test_ids, test_size=0.5, random_state=random_state
    )

    def assign_split(id):
        if id in train_ids:
            return "train"
        elif id in val_ids:
            return "val"
        else:
            return "test"

    # Assign splits for all based on patient id
    master_df = pd.DataFrame({"midas_record_id": unique_ids})
    master_df["split"] = master_df["midas_record_id"].apply(assign_split)
    check_master_split(master_df)
    export_metadata(
        master_df,
        get_data_dir()
        / "output"
        / "split_keys"
        / "master_lesion_split_lookup.xlsx",
    )

    # Assign splits for each distance based on patient id
    result_dict = {}
    for dist, subset_df in dist_dict.items():
        df = subset_df.copy()
        df["split"] = df["midas_record_id"].apply(assign_split)

        result_dict[dist] = df

        export_metadata(
            df,
            get_data_dir()
            / "output"
            / "split_keys"
            / f"{dist}_split_image_data.xlsx",
        )

    print("\nMaster lesion_key split distribution:")
    print(master_df["split"].value_counts())
    print(f"total {len(master_df)}")
    print(master_df["split"].value_counts(normalize=True))

    check_split_ratios(result_dict)
    return result_dict


result_dict = split_train_test_by_lesion(dist_dict)

✅ Master split is mutually exclusive.

Master lesion_key split distribution:
split
train    535
test      67
val       67
Name: count, dtype: int64
total 669
split
train    0.799701
test     0.100149
val      0.100149
Name: proportion, dtype: float64

1ft
Split raw counts
split
train    830
val      100
test      91
Name: count, dtype: int64
total 1021

Split proportions:
split
train    0.812929
val      0.097943
test     0.089128
Name: proportion, dtype: float64

Target proportions by split:
split  midas_path_binary
test   benign               0.516484
       malignant            0.483516
train  malignant            0.501205
       benign               0.498795
val    benign               0.570000
       malignant            0.430000
Name: proportion, dtype: float64

6in
Split raw counts
split
train    833
val      101
test      94
Name: count, dtype: int64
total 1028

Split proportions:
split
train    0.810311
val      0.098249
test     0.091440
Name: proportion, dtype: float64

Targ

In [7]:
def create_image_folders(result_dict, raw_images_dir, output_root_dir):
    """
    Creates folders and copies images into train/val/test.
    Match case-insensitive filenames (.jpg/.jpeg/_cropped.jpg/_cropped.jpeg)
    """

    raw_images_dir = Path(raw_images_dir)
    output_root_dir = Path(output_root_dir)

    all_files_map = {
        p.name.lower(): p for p in raw_images_dir.iterdir() if p.is_file()
    }

    for dist, df in result_dict.items():
        dist_dir = output_root_dir / dist

        for split in ["train", "val", "test"]:
            for label in ["benign", "malignant"]:
                (dist_dir / split / label).mkdir(parents=True, exist_ok=True)

        for row in df.itertuples(index=False):
            excel_name = row.midas_file_name.lower()
            label = row.midas_path_binary
            split = row.split

            stem = Path(excel_name).stem  # base without extension

            candidates = [
                f"{stem}.jpg",
                f"{stem}.jpeg",
                f"{stem}_cropped.jpg",
                f"{stem}_cropped.jpeg",
            ]

            matched_file = None
            for candidate in candidates:
                if candidate in all_files_map:
                    matched_file = all_files_map[candidate]
                    break

            if matched_file is None:
                print(f"No file found for: {row.midas_file_name}")
                continue

            dst_path = dist_dir / split / label / matched_file.name
            shutil.copy2(matched_file, dst_path)

    count_files_in_image_folders(output_root_dir)


create_image_folders(
    result_dict,
    get_data_dir() / "input" / "raw_images",
    get_data_dir() / "output" / "split_images",
)


=== 1ft ===
train / benign   : 414
train / malignant: 416
val   / benign   : 57
val   / malignant: 43
test  / benign   : 47
test  / malignant: 44

=== 6in ===
train / benign   : 414
train / malignant: 419
val   / benign   : 58
val   / malignant: 43
test  / benign   : 49
test  / malignant: 45

=== dscope ===
train / benign   : 413
train / malignant: 421
val   / benign   : 58
val   / malignant: 43
test  / benign   : 48
test  / malignant: 45
